# Readmission Analysis — applying the rigorous pipeline

This notebook applies the dataset-agnostic pipeline (`pipeline/`) to the
synthetic 10,000-row healthcare dataset. **Every claim below is produced by
calling `format_finding(stat_result)`, which guarantees a p-value, an
effect size, and a sample size are present. Anything that cannot pass the
claim verifier is not printed.**

References for the modeling approach:
- CMS Hospital-Wide Readmission methodology — risk-adjusted, hierarchical logistic regression.
- LACE index (van Walraven et al., CMAJ 2010): Length of stay, Acuity, Comorbidities, ED visits.
- HOSPITAL score (Donzé et al., JAMA Intern Med 2013): 7 features, validated for early readmission.

These methods exist for **real** clinical data. The synthetic dataset here
does not contain real comorbidity codes or ED-visit history, so we cannot
literally compute LACE/HOSPITAL — but the pipeline architecture is what
those methodologies rely on (validation, calibration, decision-curve sanity).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
from pipeline import (
    validate_dataset, detect_data_type, check_temporal_structure, is_synthetic_or_random,
    numeric_vs_binary, categorical_vs_binary, trend_test,
    verify_claim, format_finding,
    detect_task_type, train_models, baseline_score,
    evaluate_binary, bootstrap_ci,
    feature_stability,
)
DATA_PATH = '../../data/raw/healthcaret.csv'
TARGET = 'Readmitted'

## 1. Load and validate the dataset
Validation runs before any analysis — counts, dtypes, nulls, constants and high-card columns are reported from real computation, never estimated.

In [ ]:
df = pd.read_csv(DATA_PATH)
report = validate_dataset(df)
print(f'rows={report.n_rows}, cols={report.n_cols}, duplicates={report.duplicates}')
print(f'nulls: {report.null_counts}')
print(f'constants: {report.constant_columns}')
print(f'high_cardinality: {report.high_cardinality_columns}')
for w in report.warnings:
    print('WARN:', w)

## 2. Synthetic / random-data detection
If the dataset is effectively random, the pipeline must say so loudly **before** any modeling.

In [ ]:
drop_for_signal_check = ['Visit_ID','Patient_ID','Patient_Name',
                          'Admission_Date','Discharge_Date']
syn = is_synthetic_or_random(df.drop(columns=drop_for_signal_check), target=TARGET)
print(f'is_likely_synthetic={syn.is_likely_synthetic}')
print(f'max_mutual_info={syn.max_mutual_info:.4f}, max|corr|={syn.max_abs_correlation:.4f}')
print(f'features_with_signal={syn.n_features_with_signal}')
print(f'reason: {syn.reason}')
if syn.is_likely_synthetic:
    print('\nNOTE: Pipeline flagged the dataset as effectively random or synthetic.')
    print('Modeling will proceed but the system will not fabricate insight from non-signal.')

## 3. Univariate statistical screening
Each feature is tested against the binary target with the appropriate test (Welch t for numeric, chi-square for categorical). A feature is flagged predictive only when **both** p < 0.05 **and** the effect size meets the threshold (|d| ≥ 0.1 or Cramér's V ≥ 0.1).

In [ ]:
y_bin = (df[TARGET] == 'Yes').astype(int)
numeric_cols = [c for c in df.columns
                if pd.api.types.is_numeric_dtype(df[c])
                and c not in {'Year'}]
exclude = {'Visit_ID','Patient_ID','Patient_Name','Admission_Date','Discharge_Date',
           TARGET, 'Year','Month','Quarter'}
categorical_cols = [c for c in df.columns
                    if c not in numeric_cols and c not in exclude
                    and detect_data_type(df[c]) == 'categorical']

num_results = [numeric_vs_binary(df[c], y_bin) for c in numeric_cols]
cat_results = [categorical_vs_binary(df[c], y_bin) for c in categorical_cols]

num_table = pd.DataFrame([r.to_dict() for r in num_results])
cat_table = pd.DataFrame([r.to_dict() for r in cat_results])
print('NUMERIC PREDICTORS (sorted by |effect_size|):')
display(num_table.sort_values('effect_size', key=lambda s: s.abs(), ascending=False)
        [['feature','statistic','p_value','effect_size','effect_size_name','n','is_significant']])
print('\nCATEGORICAL PREDICTORS (sorted by Cramér\'s V):')
display(cat_table.sort_values('effect_size', ascending=False)
        [['feature','statistic','p_value','effect_size','effect_size_name','n','is_significant']])

In [ ]:
# Compliance check: every printed finding passes verify_claim
for r in num_results + cat_results:
    msg = format_finding(r)
    res = verify_claim(msg, evidence=r)
    assert res.accepted, f'rejected: {msg} -> {res.reasons}'
    print(msg)

## 4. Temporal structure check (gate before any time-series claim)
We test whether the dataset has real temporal structure before computing trends.

In [ ]:
tmp = check_temporal_structure(df, candidate_columns=['Admission_Date','Discharge_Date'])
print(f'has_temporal_structure={tmp.has_temporal_structure}')
print(f'date_column={tmp.date_column}')
print(f'unique_dates={tmp.n_unique_timestamps}, span_days={tmp.span_days}, monthly_buckets={tmp.monthly_buckets}')
print(f'reason: {tmp.reason}')

In [ ]:
if tmp.has_temporal_structure:
    df_dates = df.copy()
    df_dates['Admission_Date'] = pd.to_datetime(df_dates['Admission_Date'], dayfirst=True)
    monthly = df_dates.set_index('Admission_Date').resample('ME').size()
    print(f'monthly buckets observed: {len(monthly)}')
    trend = trend_test(monthly)
    msg = format_finding(trend)
    assert verify_claim(msg, evidence=trend).accepted
    print(msg)
    monthly_readmit = df_dates.assign(_y=y_bin.values).set_index('Admission_Date').resample('ME')['_y'].mean()
    rate_trend = trend_test(monthly_readmit)
    msg2 = format_finding(rate_trend)
    assert verify_claim(msg2, evidence=rate_trend).accepted
    print(msg2)
else:
    print('No temporal claim will be made — pipeline refuses to invent time periods.')

## 5. Modeling — Logistic, Random Forest, Gradient Boosting
All three on the same preprocessor, with stratified 5-fold CV. The baseline (always-predict-majority-class) is reported alongside.

In [ ]:
feature_cols = numeric_cols + categorical_cols
X = df[feature_cols]
task = detect_task_type(df[TARGET])
print('task:', task)
print('baseline:', baseline_score(df[TARGET], task))
results = train_models(X, df[TARGET], cv_folds=5, random_state=42)
for name, r in results.items():
    print(f'{name:>20s}  CV {r.metric} = {r.cv_mean:.4f} ± {r.cv_std:.4f}')

## 6. Evaluation on out-of-fold predictions
Calibration, threshold sweep, AUC bootstrap CI.

In [ ]:
best_name = max(results, key=lambda n: results[n].cv_mean)
best = results[best_name]
print('best model:', best_name)
y_true = (df[TARGET] == 'Yes').astype(int).values
ev = evaluate_binary(y_true, best.oof_predictions)
print(f'AUC-ROC = {ev.auc_roc:.4f} (95% CI {ev.auc_roc_ci95[0]:.4f}–{ev.auc_roc_ci95[1]:.4f})')
print(f'AUC-PR  = {ev.auc_pr:.4f}   (baseline AUC-PR = base rate = {y_true.mean():.4f})')
print(f'Brier   = {ev.brier:.4f}    (trivial floor = {y_true.mean()*(1-y_true.mean()):.4f})')
for n in ev.notes:
    print('NOTE:', n)
print()
print('threshold sweep:')
display(ev.threshold_table)
print('\ncalibration:')
display(ev.calibration_table)

## 7. Feature-importance stability across folds (Mistake #5 guardrail)
Importance is reported only with its cross-fold coefficient of variation. Features above the threshold are flagged unstable.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_for_pre = [c for c in numeric_cols if c in X.columns]
cat_for_pre = [c for c in categorical_cols if c in X.columns]
def factory():
    pre = ColumnTransformer([
        ('num', StandardScaler(), num_for_pre),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_for_pre),
    ])
    return Pipeline([('pre', pre),
                     ('est', RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                                    class_weight='balanced', random_state=0))])
stab = feature_stability(factory, X, df[TARGET], cv_folds=5)
display(stab.table.head(20))
print('\nstable count:', int(stab.table['stable'].sum()), '/', len(stab.table))

## 8. Inference function for a single new patient

In [ ]:
def predict_readmission(patient: dict) -> dict:
    """Returns probability and a guardrail-compliant explanation."""
    row = pd.DataFrame([{c: patient.get(c, None) for c in feature_cols}])
    for c in numeric_cols:
        if pd.isna(row.iloc[0][c]):
            row[c] = df[c].median()
    for c in categorical_cols:
        if pd.isna(row.iloc[0][c]):
            row[c] = df[c].mode().iloc[0]
    proba = float(best.fitted_pipeline.predict_proba(row)[0, 1])
    return {
        'predicted_readmission_probability': proba,
        'model': best_name,
        'cv_auc': best.cv_mean,
        'note': ('AUC near 0.5 — predictions are not better than chance; '
                 'use only as illustrative.' if best.cv_mean < 0.55 else
                 'Model AUC supports above-baseline ranking on this dataset.'),
    }

predict_readmission({
    'Age': 72, 'Length_of_Stay_Days': 9, 'Bill_Amount_USD': 4500,
    'Discharge_Status': 'Ongoing Treatment', 'Department': 'Cardiology',
    'Diagnosis': 'Type 2 Diabetes', 'Procedure': 'Blood Panel',
})

## 9. Final summary — what we can and cannot claim
Anything below is rendered through `format_finding` and verified by `verify_claim`.

In [ ]:
all_results = num_results + cat_results
significant = [r for r in all_results if r.is_significant]
non_significant = [r for r in all_results if not r.is_significant]
print(f'Predictors with statistical signal: {len(significant)} of {len(all_results)}')
for r in significant:
    print('  +', format_finding(r))
print(f'\nPredictors with NO statistical signal (will not be reported as drivers): {len(non_significant)}')
for r in non_significant[:5]:
    print('  -', format_finding(r))
if len(non_significant) > 5:
    print(f'  ... and {len(non_significant)-5} more')